# middle_project_ai — 위험 감지·대응 모델 학습·테스트 (Colab)

**시작 전:** 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 선택

- 코드·데이터·결과는 **구글 드라이브** `middle_project_ai_colab/middle_project_ai` 에 저장 → 세션이 끊겨도 유지
- 모델 파일은 Colab 임시 디스크에 받음 → 드라이브 용량 사용 안 함 (세션마다 다시 받지만 Colab은 빠름)
- 팀 데이터는 드라이브의 `middle_project_ai/data/custom/` CSV를 교체하면 됨

## 0. 준비

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "⚠ GPU 없음 — 런타임 유형을 T4 GPU로 바꾸세요"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
BASE = '/content/drive/MyDrive/middle_project_ai_colab'
PROJECT = f'{BASE}/middle_project_ai'
if not os.path.exists(PROJECT):
    with zipfile.ZipFile(f'{BASE}/middle_project_ai_code.zip') as z:
        z.extractall(BASE)
    print('✅ 프로젝트 압축 해제')
os.chdir(PROJECT)
print('작업 폴더:', os.getcwd())
!ls

In [ ]:
!pip install -q -r requirements-colab.txt
# (선택) Hugging Face 토큰: 왼쪽 🔑 보안 비밀에 HF_TOKEN 등록 시 사용
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

In [ ]:
!python scripts/check_env.py

## 1. F6 위험도 판단
### 1-1. 규칙 기준선

In [ ]:
!python -m src.risk.evaluate --mode rules

### 1-2. KcELECTRA 학습 → 규칙·분류기·결합 비교
샘플 데이터(24문장)라 점수는 의미 없음 — 팀 데이터로 교체 후 다시 실행

In [ ]:
!python -m src.risk.train_classifier

In [ ]:
!python -m src.risk.evaluate --mode all

In [ ]:
# 오탐 함정 세트 대신 다른 테스트셋으로 평가하려면
# !python -m src.risk.evaluate --mode all --test data/custom/risk_scenarios.csv

## 2. F4 안전 필터 (게이트 A/B)
같은 `gate_b_responses.csv`로 세 모델 비교 → 차단율·과차단율·지연시간

In [ ]:
!python -m src.guard.run_guard --model qwen0.6b

In [ ]:
!python -m src.guard.run_guard --model qwen4b

Kanana Safeguard 8B는 T4(15GB)에 그대로 안 들어가서 **4bit**로 로드 (약 6GB)

In [ ]:
!python -m src.guard.run_guard --model kanana --load_in_4bit

## 3. F5 추세 악화 탐지

In [ ]:
!python -m src.trend.trend_flag --simulate 100

## 4. F2 개인정보 마스킹

In [ ]:
!python -m src.pii.mask "저 김민수인데요 010-1234-5678로 연락 주세요. 서울시 마포구 합정동 살아요"

## 5. 결과 확인
결과는 드라이브 `outputs/`, 학습 모델은 `models/risk_classifier/`에 저장됨

In [ ]:
import glob, pandas as pd
for f in sorted(glob.glob('outputs/**/*.csv', recursive=True))[-5:]:
    print(f)
!du -sh models/* outputs 2>/dev/null

## 6. (나중에) 학습한 모델을 Hugging Face 비공개 저장소에 올리기
서비스 서버에서 이름으로 바로 받아 쓰기 위함. HF_TOKEN(write 권한) 필요

In [ ]:
# from transformers import AutoModelForSequenceClassification, AutoTokenizer
# m = AutoModelForSequenceClassification.from_pretrained('models/risk_classifier')
# t = AutoTokenizer.from_pretrained('models/risk_classifier')
# m.push_to_hub('본인HF아이디/mindbridge-risk-classifier', private=True)
# t.push_to_hub('본인HF아이디/mindbridge-risk-classifier', private=True)